In [2]:
# imports

import os
import zipfile
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import neurom as nm
from neurom.core.soma import SomaError

from scipy.stats import zscore, ttest_ind
from statsmodels.formula.api import ols
import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

import shap

sns.set(style="whitegrid")

In [3]:
# figures and CSVs saved here, can change to match local directory structure

OUTPUT_DIR = "outputs"

ZIP_CONFIGS = [
    ("/content/Control_Metadata_Compressed.zip", "/content/Control_Metadata"),
    ("/content/Control_SWC_Compressed.zip",      "/content/Control_SWC"),
    ("/content/AD_SWC_Compressed.zip",           "/content/AD_SWC"),
    ("/content/AD_Metadata_Compressed.zip",     "/content/AD_Metadata"),
]

AD_SWC_PATH      = "/content/AD_SWC/AD_SWC"
CONTROL_SWC_PATH = "/content/Control_SWC/Control_SWC"
AD_META_PATH     = "/content/AD_Metadata/AD_Metadata"
CTRL_META_PATH   = "/content/Control_Metadata/Control_Metadata"

RANDOM_SEED = 42

# global seeding
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

In [4]:
# constants

METRICS = [
    "soma_volume", "soma_surface_area", "soma_radius",
    "number_of_neurites", "number_of_sections", "number_of_segments",
    "mean_section_length", "mean_segment_length", "mean_local_bif_angle", "n_bifs",
]

METRIC_UNITS = {
    "soma_volume": "µm³",
    "soma_surface_area": "µm²",
    "soma_radius": "µm",
    "number_of_neurites": None,
    "number_of_sections": None,
    "number_of_segments": None,
    "mean_section_length": "µm",
    "mean_segment_length": "µm",
    "mean_local_bif_angle": "°",
    "n_bifs": None,
}

SEX_ORDER = ["Male", "Female"]

In [5]:
# output directory setup

os.makedirs(OUTPUT_DIR, exist_ok=True)

def out(filename: str) -> str:
    return os.path.join(OUTPUT_DIR, filename)

In [6]:
# helper functions

def extract_zip(zip_path: str, extract_to: str) -> None:
    os.makedirs(extract_to, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(extract_to)
    print(f"Extracted: {zip_path} --> {extract_to}")

def clean_metric_name(name: str) -> str:
    replacements = {
        "n_bifs": "Number of Bifurcations",
        "mean_local_bif_angle": "Mean Local Bifurcation Angle",
    }
    if name in replacements:
        return replacements[name]
    return name.replace("bif", "bifurcation").replace("_", " ").title().replace(" Of ", " of ")

def remove_outliers(df: pd.DataFrame, cols: list, threshold: float = 3.0) -> pd.DataFrame:
    z = df[cols].apply(zscore)
    return df[(z.abs() <= threshold).all(axis=1)]

def cohen_d(x: pd.Series, y: pd.Series) -> float:
    nx, ny = len(x), len(y)
    pooled_std = np.sqrt(
        ((nx - 1) * x.std(ddof=1) ** 2 + (ny - 1) * y.std(ddof=1) ** 2)
        / (nx + ny - 2)
    )
    return (x.mean() - y.mean()) / pooled_std

def make_y_label(metric: str) -> str:
    unit = METRIC_UNITS.get(metric)
    name = clean_metric_name(metric)
    return f"{name} ({unit})" if unit else name

def save_fig(fig: plt.Figure, filename: str, dpi: int = 300) -> None:
    path = out(filename)
    fig.savefig(path, dpi=dpi, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved figure: {path}")

def save_csv(df: pd.DataFrame, filename: str) -> None:
    path = out(filename)
    df.to_csv(path, index=False)
    print(f"Saved CSV:    {path}")

def run_ttests(df: pd.DataFrame, metrics: list, filename: str) -> pd.DataFrame:
    rows = []
    for metric in metrics:
        ctrl = df[df["group"] == "Control"][metric].dropna()
        ad   = df[df["group"] == "AD"][metric].dropna()

        if len(ctrl) < 2 or len(ad) < 2:
            continue

        t, p = ttest_ind(ctrl, ad, equal_var=False)
        rows.append({
            "metric": metric,
            "t_stat": t,
            "p_raw": p,
            "n_ctrl": len(ctrl),
            "n_ad": len(ad)
        })

    result_df = pd.DataFrame(rows)

    _, p_adj, _, _ = multipletests(result_df["p_raw"], method="bonferroni")
    result_df["p_adj"] = p_adj
    result_df["significant"] = result_df["p_adj"] < 0.05

    output_df = result_df.copy()
    for col in ["p_raw", "p_adj"]:
        output_df[col] = output_df[col].apply(lambda x: f"{x:.4e}" if x < 0.001 else f"{x:.6f}")

    output_df.to_csv(filename, index=False)
    return result_df

In [7]:
# data loading

# unzip raw data
for zip_path, extract_to in ZIP_CONFIGS:
    if os.path.exists(zip_path):
        extract_zip(zip_path, extract_to)
    else:
        print(f"Zip not found")

# collect SWC file paths with group labels
all_files = [
    (os.path.join(AD_SWC_PATH, f),      "AD")
    for f in sorted(os.listdir(AD_SWC_PATH)) if f.lower().endswith(".swc")
] + [
    (os.path.join(CONTROL_SWC_PATH, f), "Control")
    for f in sorted(os.listdir(CONTROL_SWC_PATH)) if f.lower().endswith(".swc")
]
print(f"Total SWC files found: {len(all_files)}")

Extracted: /content/Control_Metadata_Compressed.zip --> /content/Control_Metadata
Extracted: /content/Control_SWC_Compressed.zip --> /content/Control_SWC
Extracted: /content/AD_SWC_Compressed.zip --> /content/AD_SWC
Extracted: /content/AD_Metadata_Compressed.zip --> /content/AD_Metadata
Total SWC files found: 6035


In [8]:
# feature extraction

records, skipped = [], []

for filepath, label in all_files:
    try:
        m = nm.load_morphology(filepath)
    except (AssertionError, SomaError) as e:
        skipped.append({"name": os.path.basename(filepath), "group": label, "error": str(e)})
        continue

    # bifurcation angles are only defined at branch points with exactly 2 children
    bif_angles = [
        nm.features.bifurcation.local_bifurcation_angle(s)
        for s in nm.iter_sections(m)
        if len(s.children) == 2
    ]

    records.append({
        "neuron_name":         os.path.basename(filepath),
        "group":               label,
        "soma_volume":         nm.features.morphology.soma_volume(m),
        "soma_surface_area":   nm.features.morphology.soma_surface_area(m),
        "soma_radius":         nm.features.morphology.soma_radius(m),
        "number_of_neurites":  nm.get("number_of_neurites", m),
        "number_of_sections":  nm.get("number_of_sections", m),
        "number_of_segments":  nm.get("number_of_segments", m),
        "mean_section_length": np.mean(nm.get("section_lengths", m)) or 0,
        "mean_segment_length": np.mean(nm.get("segment_lengths", m)) or 0,
        "mean_local_bif_angle": np.mean(bif_angles) if bif_angles else 0,
        # number of bifurcations is derived from section topology
        "n_bifs":              sum(1 for s in nm.iter_sections(m) if len(s.children) == 2),
    })

print(f"Loaded: {len(records)} neurons | Skipped: {len(skipped)}")
for s in skipped:
    print(f"  [{s['group']}] {s['name']} — {s['error']}")

morpho_df = pd.DataFrame(records)

save_csv(morpho_df, "neuron_metrics_raw.csv")
save_csv(pd.DataFrame(skipped), "skipped_neurons.csv")


/usr/local/lib/python3.12/dist-packages/neurom/core/soma.py:352: UserWarning: Approximating soma volume by a sphere. <morphio._morphio.Soma object at 0x7d9630e99eb0>
  warnings.warn('Approximating soma volume by a sphere. {}'.format(morphio_soma))
/usr/local/lib/python3.12/dist-packages/neurom/core/soma.py:310: UserWarning: Approximating soma area by a sphere. <morphio._morphio.Soma object at 0x7d9630e99eb0>
  warnings.warn('Approximating soma area by a sphere. {}'.format(morphio_soma))
/usr/local/lib/python3.12/dist-packages/neurom/core/soma.py:352: UserWarning: Approximating soma volume by a sphere. <morphio._morphio.Soma object at 0x7d95e1dc5570>
  warnings.warn('Approximating soma volume by a sphere. {}'.format(morphio_soma))
/usr/local/lib/python3.12/dist-packages/neurom/core/soma.py:310: UserWarning: Approximating soma area by a sphere. <morphio._morphio.Soma object at 0x7d95e1dc5570>
  warnings.warn('Approximating soma area by a sphere. {}'.format(morphio_soma))
/usr/local/lib/p

Loaded: 6010 neurons | Skipped: 25
  [Control] 3DDS_13_VK.CNG.swc — 
/content/Control_SWC/Control_SWC/3DDS_13_VK.CNG.swc:28:error

Found soma bifurcation
The following children have been found:
/content/Control_SWC/Control_SWC/3DDS_13_VK.CNG.swc:29:warning


/content/Control_SWC/Control_SWC/3DDS_13_VK.CNG.swc:533:warning


  [Control] 3DDS_14_VK.CNG.swc — 
/content/Control_SWC/Control_SWC/3DDS_14_VK.CNG.swc:14:error

Found soma bifurcation
The following children have been found:
/content/Control_SWC/Control_SWC/3DDS_14_VK.CNG.swc:15:warning


/content/Control_SWC/Control_SWC/3DDS_14_VK.CNG.swc:42:warning


  [Control] 3DDS_15_VK.CNG.swc — 
/content/Control_SWC/Control_SWC/3DDS_15_VK.CNG.swc:3:error

Found soma bifurcation
The following children have been found:
/content/Control_SWC/Control_SWC/3DDS_15_VK.CNG.swc:4:warning


/content/Control_SWC/Control_SWC/3DDS_15_VK.CNG.swc:6:warning


  [Control] 3DDS_17_VK.CNG.swc — 
/content/Control_SWC/Control_SWC/3DDS_17_VK.CNG.swc:2:error

Found

In [9]:
# metadata integration and age matching

# metadata helper function
def load_metadata(folder_path: str, group_label: str) -> pd.DataFrame:
    """Load all per-neuron CSV metadata files from a folder into one DataFrame."""
    frames = []
    for fname in sorted(os.listdir(folder_path)):
        if fname.endswith(".csv"):
            df_tmp = pd.read_csv(os.path.join(folder_path, fname))
            df_tmp["neuron_name"] = fname.replace("_metadata.csv", ".swc")
            df_tmp["group"] = group_label
            frames.append(df_tmp)
    return pd.concat(frames, ignore_index=True)


metadata_df = pd.concat(
    [load_metadata(AD_META_PATH, "AD"), load_metadata(CTRL_META_PATH, "Control")],
    ignore_index=True,
).dropna(axis=1, how="all")

# normalizing neuron names for merging
morpho_df["neuron_name_base"] = (
    morpho_df["neuron_name"]
    .str.replace(".CNG.swc", "", regex=False)
    .str.replace(".swc",     "", regex=False)
    .str.strip()
)
metadata_df["neuron_name_base"] = (
    metadata_df["neuron_name"]
    .str.replace(".swc", "", regex=False)
    .str.strip()
)

n_shared = len(set(morpho_df["neuron_name_base"]) & set(metadata_df["neuron_name_base"]))
print(f"Shared neuron names found: {n_shared}")

merged_df = morpho_df.merge(metadata_df, on="neuron_name_base", suffixes=("_morph", "_meta"))

assert (merged_df["group_morph"] == merged_df["group_meta"]).all(), \
    "Group labels differ between morphology and metadata files!"
merged_df = merged_df.drop(columns="group_meta").rename(columns={"group_morph": "group"})

save_csv(merged_df, "neuron_metrics_with_metadata.csv")

# age matching: adult-only cohort
  # the AD group contains only adult animals, so Controls are restricted to adults only for fair comparison
  # ambiguous sex entries are excluded
adult_df = merged_df[
    (merged_df["age_classification"] == "adult") &
    (merged_df["gender"] != "Male/Female")
].copy()

print("\nAdult cohort size per group:")
print(adult_df.groupby("group").size())

save_csv(adult_df, "adult_cohort.csv")

Shared neuron names found: 6010
Saved CSV:    outputs/neuron_metrics_with_metadata.csv

Adult cohort size per group:
group
AD         1176
Control    2642
dtype: int64
Saved CSV:    outputs/adult_cohort.csv


In [10]:
# eda: full dataset

morpho_clean = remove_outliers(adult_df, METRICS)
print(f"\nAdult cohort after outlier removal: {morpho_clean.shape}")
print(morpho_clean["group"].value_counts())

# Welch t-tests: AD vs. Control (Bonferroni corrected, n = 10)
ttest_full_df = run_ttests(morpho_clean, METRICS, "ttest_adult_unbalanced.csv")
print(ttest_full_df[["metric", "t_stat", "p_raw", "p_adj", "significant"]].to_string(index=False))

# boxplots
for metric in METRICS:
    fig, ax = plt.subplots(figsize=(7, 4))
    sns.boxplot(x="group", y=metric, data=morpho_clean, palette="Set2",
                showfliers=False, ax=ax)
    ax.set(xlabel="Group", ylabel=make_y_label(metric),
           title=f"Boxplot of {clean_metric_name(metric)} by Group (Adult Cohort)")
    plt.tight_layout()
    save_fig(fig, f"boxplot_adult_{metric}.png")

# pairplot
g = sns.pairplot(morpho_clean[METRICS + ["group"]], hue="group", diag_kind="kde", corner=True)
g.figure.suptitle("Pairwise Metric Correlations (Adult Cohort)", y=1.02, fontsize=18)
save_fig(g.figure, "pairplot_adult_cohort.png")

# correlation heatmap
corr_full = morpho_clean[METRICS].corr().rename(index=clean_metric_name, columns=clean_metric_name)
save_csv(corr_full.reset_index(), "corr_matrix_adult_cohort.csv")

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr_full.astype(float), annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, vmin=-1, vmax=1, square=True,
            cbar_kws={"label": "Pearson Correlation (r)"}, ax=ax)
ax.set_title("Correlation Heatmap (Pearson r)\nAdult Cohort", fontsize=16, pad=20)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
plt.tight_layout()
save_fig(fig, "corr_heatmap_adult_cohort.png")


Adult cohort after outlier removal: (3585, 54)
group
Control    2416
AD         1169
Name: count, dtype: int64
              metric     t_stat        p_raw        p_adj  significant
         soma_volume  15.405026 2.141287e-51 2.141287e-50         True
   soma_surface_area  15.293738 5.266991e-51 5.266991e-50         True
         soma_radius   9.564464 2.028085e-21 2.028085e-20         True
  number_of_neurites   3.654055 2.632392e-04 2.632392e-03         True
  number_of_sections  20.783596 2.403164e-90 2.403164e-89         True
  number_of_segments  18.251992 5.671832e-71 5.671832e-70         True
 mean_section_length -15.912063 1.490505e-54 1.490505e-53         True
 mean_segment_length   7.205203 7.544234e-13 7.544234e-12         True
mean_local_bif_angle   8.598962 1.189041e-17 1.189041e-16         True
              n_bifs  20.751041 4.166768e-90 4.166768e-89         True


/tmp/ipykernel_843/207317153.py:14: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(x="group", y=metric, data=morpho_clean, palette="Set2",


Saved figure: outputs/boxplot_adult_soma_volume.png


/tmp/ipykernel_843/207317153.py:14: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(x="group", y=metric, data=morpho_clean, palette="Set2",


Saved figure: outputs/boxplot_adult_soma_surface_area.png


...

Saved figure: outputs/boxplot_adult_mean_segment_length.png


/tmp/ipykernel_843/207317153.py:14: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(x="group", y=metric, data=morpho_clean, palette="Set2",


Saved figure: outputs/boxplot_adult_mean_local_bif_angle.png


/tmp/ipykernel_843/207317153.py:14: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(x="group", y=metric, data=morpho_clean, palette="Set2",


Saved figure: outputs/boxplot_adult_n_bifs.png
Saved figure: outputs/pairplot_adult_cohort.png
Saved CSV:    outputs/corr_matrix_adult_cohort.csv
Saved figure: outputs/corr_heatmap_adult_cohort.png


In [11]:
# eda: class-balanced dataset
  # Control group downsampled to match AD count

ad_df      = adult_df[adult_df["group"] == "AD"]
control_df = adult_df[adult_df["group"] == "Control"]

control_downsampled = control_df.sample(n=len(ad_df), random_state=RANDOM_SEED)
balanced_df = (
    pd.concat([ad_df, control_downsampled])
    .sample(frac=1, random_state=RANDOM_SEED)
    .reset_index(drop=True)
)
balanced_df["label"] = balanced_df["group"].map({"Control": 0, "AD": 1})

balanced_clean = remove_outliers(balanced_df, METRICS)
print(f"\nBalanced adult cohort after outlier removal: {balanced_clean.shape}")
print(balanced_clean["group"].value_counts())

save_csv(balanced_clean, "neuron_metrics_balanced_adult.csv")

# pairplot
g = sns.pairplot(balanced_clean[METRICS + ["group"]], hue="group", diag_kind="kde", corner=True)
g.figure.suptitle("Pairwise Metric Correlations (Balanced Adult Cohort)", y=1.02, fontsize=18)
save_fig(g.figure, "pairplot_balanced_adult_cohort.png")

# correlation heatmap
corr_balanced = balanced_clean[METRICS].corr().rename(index=clean_metric_name, columns=clean_metric_name)
save_csv(corr_balanced.reset_index(), "corr_matrix_balanced_adult_cohort.csv")

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr_balanced.astype(float), annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, vmin=-1, vmax=1, square=True,
            cbar_kws={"label": "Pearson Correlation (r)"}, ax=ax)
ax.set_title("Correlation Heatmap (Pearson r)\nBalanced Adult Cohort", fontsize=16, pad=20)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
plt.tight_layout()
save_fig(fig, "corr_heatmap_balanced_adult_cohort.png")

# Welch t-tests: AD vs. Control (Bonferroni corrected, n = 10)
ttest_bal_df = run_ttests(balanced_clean, METRICS, "ttest_adult_balanced.csv")
print(ttest_bal_df[["metric", "t_stat", "p_raw", "p_adj", "significant"]].to_string(index=False))


Balanced adult cohort after outlier removal: (2210, 55)
group
AD         1167
Control    1043
Name: count, dtype: int64
Saved CSV:    outputs/neuron_metrics_balanced_adult.csv
Saved figure: outputs/pairplot_balanced_adult_cohort.png
Saved CSV:    outputs/corr_matrix_balanced_adult_cohort.csv
Saved figure: outputs/corr_heatmap_balanced_adult_cohort.png
              metric     t_stat        p_raw        p_adj  significant
         soma_volume  10.089786 5.890870e-23 5.890870e-22         True
   soma_surface_area  10.485498 1.083569e-24 1.083569e-23         True
         soma_radius   6.798006 1.539734e-11 1.539734e-10         True
  number_of_neurites   2.786653 5.373613e-03 5.373613e-02        False
  number_of_sections  15.737692 2.645848e-52 2.645848e-51         True
  number_of_segments  14.111100 6.270617e-43 6.270617e-42         True
 mean_section_length -13.092421 1.065067e-37 1.065067e-36         True
 mean_segment_length   4.950576 8.579143e-07 8.579143e-06         True
mean_l

In [12]:
# classification

X        = balanced_df[METRICS]
y        = balanced_df["label"]
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_df     = pd.DataFrame(X_scaled, columns=METRICS)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

# seeded background
background = shap.sample(X_df, 100, random_state=RANDOM_SEED)

# classification helper functions
def plot_confusion_matrix(y_true, y_pred, title: str, filename: str) -> None:
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["Control", "5xFAD"],
                yticklabels=["Control", "5xFAD"], ax=ax)
    ax.set(xlabel="Predicted", ylabel="Actual", title=title)
    plt.tight_layout()
    save_fig(fig, filename)

def plot_roc_curve(clf, X, y, cv, title: str, filename: str) -> float:
    y_prob = cross_val_predict(clf, X, y, cv=cv, method="predict_proba")[:, 1]
    fpr, tpr, _ = roc_curve(y, y_prob)
    roc_auc = auc(fpr, tpr)
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.plot(fpr, tpr, label=f"AUC = {roc_auc:.2f}")
    ax.plot([0, 1], [0, 1], "k--", label="Chance")
    ax.set(xlabel="False Positive Rate", ylabel="True Positive Rate", title=title)
    ax.legend(loc="lower right")
    plt.tight_layout()
    save_fig(fig, filename)
    return roc_auc

def plot_feature_importance(names, values, title: str, xlabel: str, filename: str) -> None:
    sorted_idx = np.argsort(np.abs(values))[::-1]
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.barh(np.array(names)[sorted_idx], values[sorted_idx])
    ax.set(xlabel=xlabel, title=title)
    ax.invert_yaxis()
    plt.tight_layout()
    save_fig(fig, filename)

def plot_shap_bar(explainer, X_df: pd.DataFrame, clf_name: str,
                  filename: str, class_idx: int = None) -> None:
    shap_vals = explainer(X_df, check_additivity=False)

    display_names = [clean_metric_name(m) for m in METRICS]

    if shap_vals.values.ndim == 3:
        expl = shap.Explanation(
            values=shap_vals.values[:, :, class_idx],
            base_values=shap_vals.base_values[:, class_idx],
            data=X_df,
            feature_names=display_names,
        )
    else:
        expl = shap.Explanation(
            values=shap_vals.values,
            base_values=shap_vals.base_values,
            data=X_df,
            feature_names=display_names,
        )

    ax = shap.plots.bar(expl, show=False)
    ax.set_title(f"SHAP Feature Importance – {clf_name}")
    fig = ax.figure
    fig.tight_layout()
    save_fig(fig, filename)

def evaluate_classifier(clf, name: str, X, y, cv, slug: str) -> dict:
    print(f"\n{'='*60}\n  {name}\n{'='*60}")

    y_pred     = cross_val_predict(clf, X, y, cv=cv)
    acc_scores = cross_val_score(clf, X, y, cv=cv, scoring="accuracy")

    mean_acc = acc_scores.mean()
    std_acc  = acc_scores.std()
    print(f"5-Fold CV Accuracy: {mean_acc:.3f} ± {std_acc:.3f}")
    print(classification_report(y, y_pred, target_names=["Control", "5xFAD"]))

    plot_confusion_matrix(y, y_pred,
                          title=f"Confusion Matrix – {name}",
                          filename=f"confusion_matrix_{slug}.png")

    roc_auc = plot_roc_curve(clf, X, y, cv,
                             title=f"ROC Curve – {name}",
                             filename=f"roc_curve_{slug}.png")

    return {"classifier": name, "cv_accuracy_mean": mean_acc,
            "cv_accuracy_std": std_acc, "roc_auc": roc_auc}

# logistic regression
logreg     = LogisticRegression(max_iter=1000, random_state=RANDOM_SEED)
lr_results = evaluate_classifier(logreg, "Logistic Regression", X_scaled, y, cv, slug="logreg")

logreg.fit(X_scaled, y)
plot_feature_importance(X.columns, logreg.coef_[0],
                        title="Logistic Regression Feature Coefficients",
                        xlabel="Coefficient Value",
                        filename="feature_coef_logreg.png")

lr_explainer = shap.LinearExplainer(logreg, background)
plot_shap_bar(lr_explainer, X_df, "Logistic Regression", filename="shap_bar_logreg.png")

# random forest
rf         = RandomForestClassifier(n_estimators=100, random_state=RANDOM_SEED)
rf_results = evaluate_classifier(rf, "Random Forest", X_scaled, y, cv, slug="rf")

rf.fit(X_scaled, y)
plot_feature_importance(X.columns, rf.feature_importances_,
                        title="Random Forest Feature Importances",
                        xlabel="Feature Importance",
                        filename="feature_importance_rf.png")

rf_explainer = shap.Explainer(rf, background)
plot_shap_bar(rf_explainer, X_df, "Random Forest",
              filename="shap_bar_rf.png", class_idx=1)

# gradient boosting
gb         = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=RANDOM_SEED)
gb_results = evaluate_classifier(gb, "Gradient Boosting", X_scaled, y, cv, slug="gb")

gb.fit(X_scaled, y)
plot_feature_importance(X.columns, gb.feature_importances_,
                        title="Gradient Boosting Feature Importances",
                        xlabel="Feature Importance",
                        filename="feature_importance_gb.png")

gb_explainer = shap.Explainer(gb, background)
plot_shap_bar(gb_explainer, X_df, "Gradient Boosting", filename="shap_bar_gb.png")

# save all classifier performance
save_csv(pd.DataFrame([lr_results, rf_results, gb_results]),
         "classifier_performance_summary.csv")


  Logistic Regression
5-Fold CV Accuracy: 0.716 ± 0.021
              precision    recall  f1-score   support

     Control       0.75      0.65      0.70      1176
       5xFAD       0.69      0.78      0.73      1176

    accuracy                           0.72      2352
   macro avg       0.72      0.72      0.71      2352
weighted avg       0.72      0.72      0.71      2352

Saved figure: outputs/confusion_matrix_logreg.png
Saved figure: outputs/roc_curve_logreg.png
Saved figure: outputs/feature_coef_logreg.png
Saved figure: outputs/shap_bar_logreg.png

  Random Forest
5-Fold CV Accuracy: 0.961 ± 0.011
              precision    recall  f1-score   support

     Control       0.94      0.98      0.96      1176
       5xFAD       0.98      0.94      0.96      1176

    accuracy                           0.96      2352
   macro avg       0.96      0.96      0.96      2352
weighted avg       0.96      0.96      0.96      2352

Saved figure: outputs/confusion_matrix_rf.png
Saved figur

 99%|===================| 4645/4704 [00:43<00:00]       

Saved figure: outputs/shap_bar_rf.png

  Gradient Boosting
5-Fold CV Accuracy: 0.962 ± 0.005
              precision    recall  f1-score   support

     Control       0.95      0.98      0.96      1176
       5xFAD       0.98      0.95      0.96      1176

    accuracy                           0.96      2352
   macro avg       0.96      0.96      0.96      2352
weighted avg       0.96      0.96      0.96      2352

Saved figure: outputs/confusion_matrix_gb.png
Saved figure: outputs/roc_curve_gb.png
Saved figure: outputs/feature_importance_gb.png
Saved figure: outputs/shap_bar_gb.png
Saved CSV:    outputs/classifier_performance_summary.csv


In [13]:
# pca and k-means clustering

# using separate scaler fit on the balanced, outlier-removed subset
# to avoid potential leakage from classification scaler
X_clean        = balanced_clean[METRICS]
scaler_clean   = StandardScaler()
X_clean_scaled = scaler_clean.fit_transform(X_clean)

pca        = PCA(n_components=2)
pca_coords = pca.fit_transform(X_clean_scaled)

balanced_clean = balanced_clean.copy()
balanced_clean["PCA1"] = pca_coords[:, 0]
balanced_clean["PCA2"] = pca_coords[:, 1]

# pca scatterplot
fig, ax = plt.subplots(figsize=(7, 6))
sns.scatterplot(data=balanced_clean, x="PCA1", y="PCA2", hue="group",
                palette={"Control": "skyblue", "AD": "crimson"},
                alpha=0.7, s=50, ax=ax)
ax.set_title("PCA of Microglial Morphometric Features")
plt.tight_layout()
save_fig(fig, "pca_scatter_by_group.png")

# pca loadings
loadings = pd.DataFrame(pca.components_.T, columns=["PCA1", "PCA2"], index=METRICS)
save_csv(loadings.reset_index().rename(columns={"index": "metric"}), "pca_loadings.csv")

print("\nTop 3 features for PCA1:")
print(loadings["PCA1"].abs().sort_values(ascending=False).head(3))
print("\nTop 3 features for PCA2:")
print(loadings["PCA2"].abs().sort_values(ascending=False).head(3))

#k-means helper function
def run_kmeans(X_scaled, pca_coords, group_labels, label: str) -> dict:
    kmeans   = KMeans(n_clusters=2, random_state=RANDOM_SEED, n_init=10)
    clusters = kmeans.fit_predict(X_scaled)
    sil      = silhouette_score(X_scaled, clusters)
    print(f"\nK-Means Silhouette Score ({label}): {sil:.3f}")

    slug = label.lower().replace(" ", "_")
    fig, ax = plt.subplots(figsize=(7, 6))
    sns.scatterplot(x=pca_coords[:, 0], y=pca_coords[:, 1],
                    hue=clusters.astype(str),
                    style=group_labels, palette="Set1",
                    s=60, alpha=0.7, ax=ax)
    ax.set(xlabel="PCA1", ylabel="PCA2",
           title=f"PCA Visualization of K-Means Clusters ({label})")
    ax.legend(title="Cluster / Group", fontsize=8, title_fontsize=10)
    plt.tight_layout()
    save_fig(fig, f"pca_scatter_kmeans_{slug}.png")

    return {"feature_set": label, "n_clusters": 2, "silhouette_score": sil}

# k-means: all features
kmeans_results = []

kmeans_results.append(
    run_kmeans(X_clean_scaled, pca_coords,
               group_labels=balanced_clean["group"],
               label="All Features")
)

# k-means: top pca features
top_pca1 = loadings["PCA1"].abs().sort_values(ascending=False).head(3).index.tolist()
top_pca2 = loadings["PCA2"].abs().sort_values(ascending=False).head(3).index.tolist()
top_pca_features = sorted(set(top_pca1) | set(top_pca2))
print(f"\nTop PCA features selected for K-Means: {top_pca_features}")

X_top_pca        = balanced_clean[top_pca_features]
scaler_top        = StandardScaler()
X_top_pca_scaled  = scaler_top.fit_transform(X_top_pca)

kmeans_results.append(
    run_kmeans(X_top_pca_scaled, pca_coords,
               group_labels=balanced_clean["group"],
               label="Top PCA Features")
)

save_csv(pd.DataFrame(kmeans_results), "kmeans_silhouette_scores.csv")

Saved figure: outputs/pca_scatter_by_group.png
Saved CSV:    outputs/pca_loadings.csv

Top 3 features for PCA1:
soma_surface_area    0.487131
soma_volume          0.467727
soma_radius          0.445301
Name: PCA1, dtype: float64

Top 3 features for PCA2:
n_bifs                0.544874
number_of_sections    0.543273
number_of_segments    0.517282
Name: PCA2, dtype: float64

K-Means Silhouette Score (All Features): 0.611
Saved figure: outputs/pca_scatter_kmeans_all_features.png

Top PCA features selected for K-Means: ['n_bifs', 'number_of_sections', 'number_of_segments', 'soma_radius', 'soma_surface_area', 'soma_volume']

K-Means Silhouette Score (Top PCA Features): 0.648
Saved figure: outputs/pca_scatter_kmeans_top_pca_features.png
Saved CSV:    outputs/kmeans_silhouette_scores.csv


In [14]:
# grouped plotting helper functions

def plot_grouped_boxplot(df, x, y, hue, order,
                         title, xlabel, ylabel, filename) -> None:
    fig, ax = plt.subplots(figsize=(9, 5))
    sns.boxplot(data=df, x=x, y=y, hue=hue, order=order,
                showfliers=False, palette="Set2", ax=ax)
    ax.set(title=title, xlabel=xlabel, ylabel=ylabel)
    ax.legend(fontsize=10, title_fontsize=12)
    plt.tight_layout()
    save_fig(fig, filename)


def plot_grouped_barplot(df, x, y, hue, order,
                         title, xlabel, ylabel, filename) -> None:
    avg_df = df.groupby([x, hue])[y].mean().reset_index()
    fig, ax = plt.subplots(figsize=(9, 5))
    sns.barplot(data=avg_df, x=x, y=y, hue=hue, order=order, palette="Set2", ax=ax)
    ax.set(title=title, xlabel=xlabel, ylabel=ylabel)
    ax.legend(fontsize=10, title_fontsize=12)
    plt.tight_layout()
    save_fig(fig, filename)

In [15]:
# subgroup analysis: sex

# ambiguous sex entries excluded
sex_df = morpho_clean[morpho_clean["gender"].isin(SEX_ORDER)].copy()

for metric in METRICS:
    label = make_y_label(metric)
    clean = clean_metric_name(metric)

    plot_grouped_boxplot(
        sex_df, x="gender", y=metric, hue="group",
        order=SEX_ORDER,
        title=f"Boxplot of {clean} by Sex and Group",
        xlabel="Sex", ylabel=label,
        filename=f"boxplot_sex_{metric}.png",
    )
    plot_grouped_barplot(
        sex_df, x="gender", y=metric, hue="group",
        order=SEX_ORDER,
        title=f"Average {clean} by Sex and Group",
        xlabel="Sex", ylabel=f"Average {label}",
        filename=f"barplot_sex_{metric}.png",
    )

# sex-stratified Welch t-tests (Bonferroni correction, n = 20)
# family = 10 morphometric features x 2 sexes, corrected together
sex_ttest_rows = []
for sex in SEX_ORDER:
    for metric in METRICS:
        ad_vals   = sex_df[(sex_df["group"] == "AD")      & (sex_df["gender"] == sex)][metric].dropna()
        ctrl_vals = sex_df[(sex_df["group"] == "Control") & (sex_df["gender"] == sex)][metric].dropna()
        if len(ad_vals) >= 2 and len(ctrl_vals) >= 2:
            t_stat, p_raw = ttest_ind(ad_vals, ctrl_vals, equal_var=False)
            sex_ttest_rows.append({
                "sex": sex, "metric": metric,
                "t_stat": t_stat, "p_raw": p_raw,
                "n_ad": len(ad_vals), "mean_ad": ad_vals.mean(), "std_ad": ad_vals.std(),
                "n_ctrl": len(ctrl_vals), "mean_ctrl": ctrl_vals.mean(), "std_ctrl": ctrl_vals.std(),
            })

sex_ttest_df = pd.DataFrame(sex_ttest_rows)
_, sex_ttest_df["p_adj"], _, _ = multipletests(sex_ttest_df["p_raw"], method="bonferroni")
sex_ttest_df["metric_clean"] = sex_ttest_df["metric"].apply(clean_metric_name)
save_csv(sex_ttest_df, "ttest_by_sex.csv")

# −log10(adj. p) heatmap
pval_pivot = sex_ttest_df.pivot(index="metric_clean", columns="sex", values="p_adj")
log_pvals  = -np.log10(pval_pivot.replace(0, 1e-300))
log_pvals  = log_pvals.round(3) + 0.0

fig, ax = plt.subplots(figsize=(14, 8))
sns.heatmap(
    log_pvals,
    annot=True,
    fmt=".3f",
    cmap="coolwarm",
    linewidths=0.5,
    linecolor="white",
    square=False,
    cbar_kws={
        "label": r"$-\log_{10}(\text{p}_{\text{adj}})$",
        "shrink": 0.8
    },
    ax=ax
)
# standardized layout settings
ax.set_title("T-test: AD vs. Control by Sex and Metric (Bonferroni Corrected)", fontsize=14, pad=16)
ax.set_xlabel("Sex", fontsize=12, labelpad=10)
ax.set_ylabel("Metric", fontsize=12, labelpad=10)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right", fontsize=11)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=11)
plt.tight_layout()
save_fig(fig, "ttest_sex_pvalue_heatmap.png")

Saved figure: outputs/boxplot_sex_soma_volume.png
Saved figure: outputs/barplot_sex_soma_volume.png
Saved figure: outputs/boxplot_sex_soma_surface_area.png
Saved figure: outputs/barplot_sex_soma_surface_area.png
Saved figure: outputs/boxplot_sex_soma_radius.png
Saved figure: outputs/barplot_sex_soma_radius.png
Saved figure: outputs/boxplot_sex_number_of_neurites.png
Saved figure: outputs/barplot_sex_number_of_neurites.png
Saved figure: outputs/boxplot_sex_number_of_sections.png
Saved figure: outputs/barplot_sex_number_of_sections.png
Saved figure: outputs/boxplot_sex_number_of_segments.png
Saved figure: outputs/barplot_sex_number_of_segments.png
Saved figure: outputs/boxplot_sex_mean_section_length.png
Saved figure: outputs/barplot_sex_mean_section_length.png
Saved figure: outputs/boxplot_sex_mean_segment_length.png
Saved figure: outputs/barplot_sex_mean_segment_length.png
Saved figure: outputs/boxplot_sex_mean_local_bif_angle.png
Saved figure: outputs/barplot_sex_mean_local_bif_angle.

In [16]:
# two-way ANOVA: group x sex
anova_rows = []
for metric in METRICS:
    df_tmp = sex_df[["group", "gender", metric]].dropna()
    if df_tmp["group"].nunique() < 2 or df_tmp["gender"].nunique() < 2:
        continue
    try:
        model     = ols(f'Q("{metric}") ~ C(group) * C(gender)', data=df_tmp).fit()
        anova_tbl = sm.stats.anova_lm(model, typ=2)
        anova_rows.append({
            "metric":            metric,
            "F_group":           anova_tbl.loc["C(group)",           "F"],
            "p_group_raw":       anova_tbl.loc["C(group)",           "PR(>F)"],
            "F_sex":             anova_tbl.loc["C(gender)",          "F"],
            "p_sex_raw":         anova_tbl.loc["C(gender)",          "PR(>F)"],
            "F_interaction":     anova_tbl.loc["C(group):C(gender)", "F"],
            "p_interaction_raw": anova_tbl.loc["C(group):C(gender)", "PR(>F)"],
        })
    except Exception as e:
        print(f"Skipping {metric}: {e}")

anova_df = pd.DataFrame(anova_rows)

# Bonferroni correction applied independently per ANOVA factor (n = 10 metrics)
for factor in ["group", "sex", "interaction"]:
    _, anova_df[f"p_{factor}_adj"], _, _ = multipletests(
        anova_df[f"p_{factor}_raw"], method="bonferroni"
    )

save_csv(anova_df, "anova_group_x_sex.csv")

# significance barplot
melted = pd.melt(
    anova_df,
    id_vars=["metric"],
    value_vars=["p_group_adj", "p_sex_adj", "p_interaction_adj"],
    var_name="factor", value_name="p_adj",
)
# explicit mapping
factor_map = {
    "p_group_adj": "Group",
    "p_sex_adj": "Sex",
    "p_interaction_adj": "Interaction",
}
melted["factor"]       = melted["factor"].map(factor_map)
melted["-log10(p)"]    = -np.log10(melted["p_adj"].replace(0, 1e-300))
melted["metric_clean"] = melted["metric"].apply(clean_metric_name)

fig, ax = plt.subplots(figsize=(14, 6))
sns.barplot(data=melted, x="metric_clean", y="-log10(p)", hue="factor", palette="muted", ax=ax)
ax.axhline(-np.log10(0.05), color="red", linestyle="--", label="p = 0.05 threshold")
ax.set(xlabel="Morphological Metric",
       ylabel=r"$-\log_{10}(\text{p}_{\text{adj}})$",
       title="ANOVA Significance: Group, Sex, and Interaction (Bonferroni Corrected)")
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
ax.legend(title="ANOVA Factor")
plt.tight_layout()
save_fig(fig, "anova_significance_barplot.png")

Saved CSV:    outputs/anova_group_x_sex.csv


/tmp/ipykernel_843/4146879367.py:55: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")


Saved figure: outputs/anova_significance_barplot.png


In [17]:
# summary table: AD vs. Control by sex (Bonferroni across all 20 tests)

summary_rows = []
for sex in SEX_ORDER:
    for metric in METRICS:
        ctrl_vals = morpho_clean[(morpho_clean["group"] == "Control") &
                                 (morpho_clean["gender"] == sex)][metric].dropna()
        ad_vals   = morpho_clean[(morpho_clean["group"] == "AD") &
                                 (morpho_clean["gender"] == sex)][metric].dropna()
        if len(ctrl_vals) == 0 or len(ad_vals) == 0:
            continue
        t_stat, p_raw = ttest_ind(ad_vals, ctrl_vals, equal_var=False)
        pct_change    = (ad_vals.mean() - ctrl_vals.mean()) / ctrl_vals.mean() * 100
        summary_rows.append({
            "Metric":                         clean_metric_name(metric),
            "Sex":                            sex,
            "Control Mean ± SD":              f"{ctrl_vals.mean():.4f} ± {ctrl_vals.std():.4f}",
            "AD Mean ± SD":                   f"{ad_vals.mean():.4f} ± {ad_vals.std():.4f}",
            "% Change from Control":          round(pct_change, 2),
            "t-stat":                         round(t_stat, 4),
            "p-value":                        p_raw,
            "_p_raw":                         p_raw,   # temporary column for correction
            "Cohen's d":                      round(cohen_d(ad_vals, ctrl_vals), 4),
        })

summary_df = pd.DataFrame(summary_rows)

# Bonferroni across all tests in the table (20 total: 10 metrics x 2 sexes)
_, summary_df["Adjusted p-value (Bonferroni)"], _, _ = multipletests(
    summary_df["_p_raw"], method="bonferroni"
)
summary_df = summary_df.drop(columns="_p_raw")

summary_df = summary_df[[
    "Metric", "Sex", "Control Mean ± SD", "AD Mean ± SD",
    "% Change from Control", "t-stat", "p-value",
    "Adjusted p-value (Bonferroni)", "Cohen's d"
]]

summary_df["p-value"] = summary_df["p-value"].apply(
    lambda x: f"{x:.4e}" if x < 0.001 else f"{x:.6f}"
)
summary_df["Adjusted p-value (Bonferroni)"] = summary_df["Adjusted p-value (Bonferroni)"].apply(
    lambda x: f"{x:.4e}" if x < 0.001 else f"{x:.6f}"
)

save_csv(summary_df, "summary_ad_vs_control_by_sex.csv")

print(f"\n All outputs saved to: {os.path.abspath(OUTPUT_DIR)}/")


Saved CSV:    outputs/summary_ad_vs_control_by_sex.csv

 All outputs saved to: /content/outputs/
